# 按任务ID下载属性库

In [3]:
import requests
import time
from pathlib import Path
import pyperclip
import json
import os

task_id = pyperclip.paste()

assert task_id.isdigit(), "请复制任务id到剪贴板"
print("task_id =", task_id)
# host = "pre-adc-pc.alibaba-inc.com"   # 预发环境
host = "adc-pc.alibaba-inc.com"   # 生产环境

url = f"http://{host}/task/getGamePropertiesByTaskId"
payload = {"taskId": task_id, "timestamp": int(time.time()*1000)}
response = requests.post(url, json=payload)

json_path = Path().resolve() / "properties_remote.json"
if json_path.exists(): os.remove(json_path)
json_str = json.dumps(json.loads(response.json()["data"]["gameProperties"]), indent=4, ensure_ascii=False)
json_path.write_text(json_str)

task_id = 9367131


1844

# 按CID下载属性库

In [2]:
import requests
import time
from pathlib import Path
import pyperclip
import json
import os
import gzip
import base64
from io import BytesIO

categoryId = pyperclip.paste()

assert categoryId.isdigit(), "请复制cid到剪贴板"
print("categoryId =", categoryId)
# host = "pre-adc-pc.alibaba-inc.com"   # 预发环境
host = "adc-pc.alibaba-inc.com"   # 生产环境

url = f"http://{host}/task/queryGamePropertyByCid"
payload = {"cid": categoryId}
response = requests.post(url, json=payload)

def ungzip_text(text: str) -> str:
    if not text or text.isspace():
        return None
    with BytesIO(base64.b64decode(text)) as bytes_in:
        with gzip.GzipFile(fileobj=bytes_in, mode='rb') as gzip_in:
            decompressed_bytes = gzip_in.read()
            return decompressed_bytes.decode('utf-8')

json_path = Path().resolve() / "properties_remote.json"
if json_path.exists(): os.remove(json_path)

propertyValuesCompressed = json.loads(response.json()["data"])["result"]["gamePropertiesCompressed"]
json_str = json.dumps(json.loads(ungzip_text(propertyValuesCompressed)), indent=4, ensure_ascii=False)
json_path.write_text(json_str)

categoryId = 1676881636794172


TypeError: the JSON object must be str, bytes or bytearray, not NoneType

# 本地采集属性原始数据

In [7]:
import json
import shelve
from pathlib import Path
from pprint import pprint
import yaml

shelve_path = str(Path().resolve() / "properties_data")

with shelve.open(shelve_path) as properties_data:
    properties_dict = dict(properties_data)
    properties_json = json.dumps(properties_dict, indent=4, ensure_ascii=False)
    (Path().resolve() / "properties_data_local.json").write_text(properties_json)
    print(yaml.dump(properties_dict, allow_unicode=True).strip())

五星武将数: '12'
武将:
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-1_1756541142070134.jpg
  五星武将: 吴孙坚
  名称: 孙坚
  稀有度: 5星
  红度: 0红
  赛季: 其他
  阵营: 吴
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-2_1756541143436994.jpg
  五星武将: 吴孙坚
  名称: 孙坚
  稀有度: 5星
  红度: 0红
  赛季: 其他
  阵营: 吴
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-3_1756541144820336.jpg
  五星武将: 汉灵帝
  名称: 灵帝
  稀有度: 5星
  红度: 0红
  赛季: 其他
  阵营: 汉
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-4_1756541146170728.jpg
  五星武将: 汉袁绍
  名称: 汉袁绍
  稀有度: 5星
  红度: 0红
  赛季: 其他
  阵营: 汉
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-5_1756541147497925.jpg
  五星武将: 汉卢植
  名称: 卢植
  稀有度: 5星
  红度: 0红
  赛季: 其他
  阵营: 汉
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-6_1756541148844030.jpg
  五星武将: 汉卢植
  名称: 卢植
  稀有度: 5星
  红度: 0红
  赛季: 其他
  阵营: 汉
- image: /jym-oss/resource/vmos/20250830/adc_1753743/upload_WuJiang-7_1756541150203587.jpg
  五星

# 线上回传属性解压

In [10]:
import gzip
import base64
from io import BytesIO
import json
import pyperclip
from pathlib import Path

def gzip_text(text: str) -> str:
    if not text or text.isspace():
        return None

    with BytesIO() as bytes_out:
        with gzip.GzipFile(fileobj=bytes_out, mode='wb') as gzip_out:
            gzip_out.write(text.encode('utf-8'))
        compressed_bytes = bytes_out.getvalue()
        return base64.b64encode(compressed_bytes).decode('ascii')

def ungzip_text(text: str) -> str:
    if not text or text.isspace():
        return None
    with BytesIO(base64.b64decode(text)) as bytes_in:
        with gzip.GzipFile(fileobj=bytes_in, mode='rb') as gzip_in:
            decompressed_bytes = gzip_in.read()
            return decompressed_bytes.decode('utf-8')

# propertyValuesCompressed = pyperclip.paste()
propertyValuesCompressed = json.loads(pyperclip.paste())["propertyValuesCompressed"]
json_str = json.dumps(json.loads(ungzip_text(propertyValuesCompressed)), indent=4, ensure_ascii=False)
(Path().resolve() / "properties_callback_online.json").write_text(json_str)

98016

# 线上属性库更新

In [6]:
from pathlib import Path
import json

propertiesUpdateTemplates = {}
json_path = Path().resolve() / "propertiesUpdateTemplates.json"
if json_path.exists():
    json_data = json.loads(json_path.read_text())
    for property in json_data:
        propertiesUpdateTemplates[property["propertyName"]] = property

propertiesUpdateTemplates

{'装备': {'childNodes': [{'propertyName': '部位', 'propertyValueName': '衣服'},
   {'propertyName': '是否神韵', 'propertyValueName': '是'},
   {'propertyName': '宝石是否14', 'propertyValueName': '是'},
   {'propertyName': '红度', 'propertyValueName': '3红'},
   {'propertyName': '红度', 'propertyValueName': '2红'},
   {'propertyName': '部位', 'propertyValueName': '项链'},
   {'propertyName': '是否神韵', 'propertyValueName': '否'},
   {'propertyName': '宝石是否14', 'propertyValueName': '否'},
   {'propertyName': '红度', 'propertyValueName': '1红'},
   {'propertyName': '部位', 'propertyValueName': '腰带'},
   {'propertyName': '部位', 'propertyValueName': '发饰'},
   {'propertyName': '部位', 'propertyValueName': '鞋子'}],
  'propertyName': '装备',
  'propertyValueName': ''},
 '妖灵': {'childNodes': [{'propertyName': '品级', 'propertyValueName': '神兽'},
   {'propertyName': '品级', 'propertyValueName': '珍兽'},
   {'propertyName': '品级', 'propertyValueName': '太一'},
   {'propertyName': '品级', 'propertyValueName': '半仙'},
   {'propertyName': '品级', 'property

# 线上业务串流

In [ ]:
import time
from pathlib import Path
import os

import yaml
from jsonpath import jsonpath
import requests

class OnlineTest:
    def __init__(self):
        taskType = 2  # 发布2，验号3
        # host = "test-adc-pc.alibaba.net"   # 日常环境
        host = "pre-adc-pc.alibaba-inc.com"  # 预发环境
        # host = "adc-pc.alibaba-inc.com"      # 生产环境

        self.timeStamp = lambda: str(int(time.time() * 10 ** 3))
        self.submitTask_url = f"https://{host}/task/submitTask"
        self.taskstream_url = f"https://{host}/taskstream?taskId="
        gameId_platform = os.path.split(Path().resolve())[1].split("_")
        self.params = dict(gameId=gameId_platform[0], taskType=taskType, gamePlatform=gameId_platform[1], taskStatus=0, deviceInfo="", scriptId="", resultInfo="")
        self.session = requests.session()
        headers = {"User-Agent": "Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Mobile Safari/537.36"}
        self.session.headers.update(headers)
        with open(Path().resolve() / "onlinetest_config.yaml", mode='r', encoding="utf8") as yaml_file:
            self.account = yaml.load(yaml_file, Loader=yaml.FullLoader)

    def __call__(self, *args, **kwds):
        for item in self.account:
            self.params.update(userAccount=item["userAccount"])
            self.params.update(password=item["password"])
            self.params.update(gameArea=item["gameArea"])
            self.params.update(game_area=item["gameArea"])
            self.params.update(bizId=f"test{self.timeStamp()}")
            res = self.session.get(self.submitTask_url, params=self.params)
            if taskId := jsonpath(res.json(), "$..taskId"):
                output = [item["comment"], item["userAccount"], item["gameArea"], f"{self.taskstream_url}{taskId[0]}"]
                print(*output)
            time.sleep(2)

OnlineTest()()

大号2 13810507654 龙脉宝域11服 https://pre-adc-pc.alibaba-inc.com/taskstream?taskId=5585591


# 文件重命名

In [32]:
import time
from glob import glob
import os
OPJ = os.path.join
OP = os.path

timeStamp = lambda: str(int(time.time() * 10 ** 6))

class ScreenshotRename:
    def __init__(self):
        self.filesList = glob(OPJ("/Users/vip/androidLogResult", "adc_result_*_$*.jpg"))

    def filename_split(self, old_filename):
        filename = OP.splitext(old_filename)[0]
        filename_split_list = filename.split("_")
        s1, s2 = filename_split_list[-2:]
        serial = int(s1) if s1.isdigit() else 0
        id_str = s2.strip("$")
        id_int = int(id_str) if id_str.isdigit() else 0
        new_filename = f"{'_'.join(filename_split_list[:-1])}"
        return old_filename, new_filename, id_int, serial

    def online(self, filesList_order):
        for old_filename, new_filename, *args in filesList_order:
            ext = OP.splitext(old_filename)[1]
            new_filename += f"_{timeStamp()}{ext}"
            os.rename(old_filename, new_filename)
            time.sleep(0.1)

    def offline(self, filesList_order):
        for old_filename, new_filename, id_int, serial in filesList_order:
            path, name = OP.split(new_filename)
            name_split = name.split("_")
            if name_split[-1].isdigit(): name = "_".join(name_split[:-1])
            ext = OP.splitext(old_filename)[1]
            new_filename = OPJ(path, f"{id_int}_{serial}_{name}{ext}")
            os.rename(old_filename, new_filename)

    def __call__(self, *args, **kwds):
        filesList_order = list(map(self.filename_split, self.filesList))
        filesList_order.sort(key=lambda item: item[-2:])
        self.offline(filesList_order)
        # self.online(filesList_order)

ScreenshotRename()()

# JSONC转YAML

In [ ]:
# from jsonc_parser.parser import JsoncParser
# import yaml

# json_data = JsoncParser.parse_file(Path().resolve() / "onlinetest_config.jsonc")
# yaml_path = Path().resolve() / "onlinetest_config.yaml"
# with open(yaml_path, mode='w', encoding="utf8") as yaml_file:
#     yaml.dump(json_data, yaml_file, allow_unicode=True, sort_keys=False, indent=4)

# JSON转YAML

In [2]:
import json
import yaml
from pprint import pprint
from pathlib import Path

json_path = Path().resolve() / "rectify.json"
yaml_path = Path().resolve() / "rectify.yaml"

with (open(json_path, mode="r", encoding="utf8") as json_file,
    open(yaml_path, mode="w", encoding="utf8") as yaml_file):
    json_data = json.load(json_file)
    yaml.dump(json_data, yaml_file, allow_unicode=True, sort_keys=False, indent=4)

with open(yaml_path, mode='r', encoding="utf8") as yaml_file:
    pprint(yaml.load(yaml_file, Loader=yaml.FullLoader))

{'name_rectify': {'E': '山',
                  '件': '吽',
                  '减默德克萨斯': '缄默德克萨斯',
                  '子': '孑',
                  '岁': '夕',
                  '影': '傀影',
                  '愧影': '傀影',
                  '森': '森蚺',
                  '灌尘芙蓉': '濯尘芙蓉',
                  '灰炬': '灰烬',
                  '灰炽': '灰烬',
                  '灰烟': '灰烬',
                  '白面': '白面鸮',
                  '白面号': '白面鸮',
                  '白面鸡': '白面鸮',
                  '石': '燧石',
                  '砂': '慑砂',
                  '碱默德克萨斯': '缄默德克萨斯',
                  '纯炽艾雅法拉': '纯烬艾雅法拉',
                  '纯烟艾雅法拉': '纯烬艾雅法拉',
                  '纯艾雅法拉': '纯烬艾雅法拉',
                  '罗比拉塔': '罗比菈塔',
                  '镊砂': '慑砂',
                  '青积': '青枳',
                  '默德克萨斯': '缄默德克萨斯'}}


# 小工具

In [7]:
# Scrcpy 窗口缩放计算
import math

for i in range(900, 1000):
    a = 1350 * i / 2400
    if not math.modf(a)[0]:
        print(i, int(a))

912 513
928 522
944 531
960 540
976 549
992 558


In [ ]:
# 分页
data = ["a", "b", "c", "d", "e", "f", "g", "h"]
page_size = 3  # 每页的数量

import math

for page_num in range(1, math.ceil(len(data) / page_size) + 1):
    start_index = (page_num - 1) * page_size
    end_index = page_num * page_size
    print(data[start_index:end_index])

In [ ]:
# 三角函数
import math

def Joystick(centre, radius, angle):
    x = centre[0] + radius * math.cos(math.radians(angle))
    y = centre[1] + radius * math.sin(math.radians(angle))
    print(f"x={int(x)}, y={int(y)}")

Joystick((100, 100), 50, -10)

In [91]:
evo = dict.fromkeys(["SSS+数量", "SSS数量", "ss数量", "s数量"], 0)
dict(map(lambda item: (item[0], str(item[1])), evo.items()))

{'SSS+数量': '0', 'SSS数量': '0', 'ss数量': '0', 's数量': '0'}

In [2]:
from enum import Enum
class Weekday(Enum):
    monday = 1
    tuesday = 2
    wednesday = 3
    thirsday = 4
    friday = 5
    saturday = 6
    sunday = 7

print(Weekday.wednesday == "wednesday")         # Weekday.wednesday      
print(type(Weekday.wednesday))   # <enum 'Weekday'>
print(Weekday.wednesday.name)    # wednesday
print(Weekday.wednesday.value)

isinstance(Weekday.wednesday, Enum)

In [4]:
import json

with open("properties_callback_online.json") as f:
    properties = json.load(f)

for i in properties["gameProperties"]:
    print(i["value"], f"https://static-cdn.jiaoyimao.com/{i['image']}")

In [ ]:
from glob import glob
from pathlib import Path
import re
from biz_tools import select_numbers


filepath_pattern = re.compile(r"wujiang(\d+).jpg")

wujiang_images_path = glob(str(Path("/Users/vip/androidLogResult").resolve() / "wujiang*.jpg"))
wujiang_images_path = filter(lambda x: filepath_pattern.search(x), wujiang_images_path)
print(sorted(wujiang_images_path, key=select_numbers))

# path = sorted(, key=select_numbers)
# print(filepath_pattern.findall(path))

['/Users/vip/androidLogResult/wujiang0.jpg', '/Users/vip/androidLogResult/wujiang999.jpg']


# 属性数据后台批量操作

In [21]:
# 拉取属性数据列表
import requests
import json

# host = "test-adc-pc.alibaba.net"   # 日常环境
# host = "pre-adc-pc.alibaba-inc.com"  # 预发环境
host = "adc-pc.alibaba-inc.com"      # 生产环境

session = requests.session()
session.verify = False
requests.packages.urllib3.disable_warnings()

# 属性数据过滤
def property_filter(x):
    if x["game_id"] == game_id and \
        (x["id"] <= id_max or id_max == 0) and \
        x["status"] == 0 and x["id"] not in exclude:
        json_data = json.loads(x["delta_game_data"])
        propertyName = json_data[0]["propertyName"]
        propertyValueName = json_data[0]["propertyValueName"]
        print(f"{x['id']} {propertyName}: {propertyValueName}")
        return True

game_id_list = [
    # "13765",
    # "15574",
    # "2717",
    # "6586",
    "8788",
]
id_max = 0
exclude = [
    16377
]

dataIdList = []
for index, game_id in enumerate(game_id_list):
    url = f"https://{host}/task/getGamePropertyDataList"
    payload = {"current": 1, "pageSize": 100, "game_id": game_id, "status": "0"}
    response = session.post(url, json=payload)
    propertyDataList = response.json()
    data = propertyDataList["result"]["data"]
    dataIdList = sorted(map(lambda x: x["id"], (filter(property_filter, data["content"]))), reverse=True)
    if not data['total']: continue
    if dataIdList:
        print(f"{game_id} 总数：{len(dataIdList)}")
        break

62896 六星干员: 司霆惊蛰
54911 五星干员: YHIRED OPERATOR
54742 五星干员: 士THIRED OPERATORNEWLYI
54545 五星干员: ORNEWLYHIREDOPERA
54447 六星干员: LYHIREDOPERATOI
54427 六星干员: LYHIREDOPERATORNEV
54370 五星干员: ATORNEWLYHIREDOPER
54195 六星干员: RATOR NEWLYHIREDOPEI
54164 五星干员: IREDOPERATORNEWLY
53704 五星干员: 三锡兰
53626 五星干员: RATORNEWLYHIREDOPEI
53614 五星干员: REDOPERATOR NEWLYHI
53515 五星干员: ORNEWLYHIREDOPERAT
53459 六星干员: ATORNEWLYHIRED OPER
53364 五星干员: 艾丽妮
53356 五星干员: PERATOR NEWLYHIRED
53256 六星干员: YHIRED OPERATOR NEWL
53238 五星干员: ORNEWLYHIRED OPERAT
46553 五星干员: 惊
46552 五星干员: 叶
44628 六星干员: DOPERATOR NEWLYHIR
43781 六星干员: RATOR NEWLYHIREDO
42909 六星干员: 妮笑
42908 六星干员: 维娜，维多利亚
42907 五星干员: 刺政
42906 五星干员: 野紫
42905 五星干员: 截度
42904 五星干员: 冰眼
42903 五星干员: 燃石
42902 五星干员: 龙吾兰
42901 五星干员: 万项
42835 六星干员: 迷选香
42834 六星干员: 歌蕾带娅
42833 六星干员: 阳
42832 六星干员: 焰影羊草
42831 六星干员: MonЗtr
42830 六星干员: 泰
42829 六星干员: 森蛇
42828 六星干员: 鸟尔比安
42827 六星干员: 嗟峨
42826 五星干员: 香仁
42825 五星干员: 鸟有
42824 五星干员: 行署
42823 五星干员: 夏标
42822 五星干员: 刺攻
42821 五星干员: 斯拉
42820 五星干员: 炎狱炎焙
4

In [15]:
# 更新属性数据后台状态
from time import sleep

url = f"https://{host}/task/updateGamePropertyDataStatus"
for data_id in dataIdList:
    payload = {"id": data_id, "status": 2}
    response = session.post(url, json=payload).json()["result"]["data"]
    print(f"id: {response['id']}, status: {response['status']}")
    sleep(0.25)

id: 61784, status: 2
id: 61783, status: 2
id: 61580, status: 2
id: 61516, status: 2
id: 61337, status: 2
id: 61090, status: 2
id: 61089, status: 2
id: 61087, status: 2
id: 61048, status: 2
id: 60824, status: 2
id: 60621, status: 2
id: 60477, status: 2
id: 60476, status: 2
id: 60475, status: 2
id: 60210, status: 2
id: 60209, status: 2
id: 59536, status: 2
id: 59500, status: 2
id: 58872, status: 2
id: 58871, status: 2
id: 58870, status: 2
id: 58709, status: 2
id: 58647, status: 2
id: 56450, status: 2
id: 56449, status: 2
id: 56448, status: 2
id: 56447, status: 2
id: 56446, status: 2
id: 56436, status: 2
id: 53071, status: 2
id: 52310, status: 2
id: 52309, status: 2
id: 46833, status: 2
id: 46832, status: 2
id: 46831, status: 2
id: 46298, status: 2
id: 46268, status: 2
id: 46267, status: 2
id: 46145, status: 2
id: 46144, status: 2


KeyboardInterrupt: 

In [1]:
from contextlib import contextmanager

@contextmanager
def my_context():
    # 进入上下文前的操作
    print('entering context')
    try:
        yield  # 生成器函数的 `yield` 语句之前的代码作为上文，之后的代码作为下文
    finally:
        # 离开上下文后的操作
        print('exiting context')

# 使用上下文管理器
with my_context():
    print('inside context')

entering context
inside context
exiting context
